# Evaluate similarity suggestions

In [1]:
%load_ext autoreload

In [2]:
import os
import pickle
from os.path import join
from IPython.display import display, Markdown, display_html

import scanpy as sc
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt

In [3]:
%autoreload
from sconnect.dataset_ot import DatasetMapping

## Load and preprocess data

In [4]:
DATA_PATH = "/vol/data/dataset-similarity/preprocessed"
MODEL_DIR = "/vol/data/dataset-similarity/models/similarityOT"
CACHE_DIR = "/vol/data/dataset-similarity/cache"
FIG_DIR = "/vol/data/dataset-similarity/figures"


QUERY_DATASET = "f6c50495-3361-40ed-a819-fb9644396ed9"
REF_DATASET = "ced320a1-29f3-47c1-a735-513c7084d508"
N_TOP_GENES = 750

In [5]:
cache_file = join(
    CACHE_DIR, 
    "+".join([QUERY_DATASET, REF_DATASET, f"{N_TOP_GENES}HVGs"]) + ".pickle"
)
with open(cache_file, "rb") as f:
    adata_query, adata_ref = pickle.load(f)

In [6]:
adata_query

AnnData object with n_obs × n_vars = 263159 × 750
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'AuthorCellType', 'cell_type_author', 'sample_id'
    var: 'feature_id'
    uns: 'de_res_ova', 'de_res_ava'
    obsm: 'X_scTab', 'X_scimilarity'
    varm: 'de_res'

In [7]:
adata_ref

AnnData object with n_obs × n_vars = 1058909 × 750
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'author_cell_type', 'sample_uuid', 'cell_type_author', 'sample_id'
    var: 'feature_id'
    uns: 'de_res_ova', 'de_res_ava'
    obsm: 'X_scTab', 'X_scimilarity'
    varm: 'de_res'

In [8]:
MODEL_VERSION = (
    f"{QUERY_DATASET}+{REF_DATASET}"
    "+n_top_genes=750"
    "+n_genes_query_ova=10"
    "+n_genes_ref_ova=20"
    "+tau=1.0"
    "+epsilon=0.05"
    "+embedding_layer=None"
)

mapping = pd.read_parquet(join(MODEL_DIR, MODEL_VERSION, "mapping_mean.parquet"))
distance = pd.read_parquet(join(MODEL_DIR, MODEL_VERSION, "distance.parquet"))

In [9]:
def extract_ontology_mapping(adata):
    return (
        adata.obs[["cell_type_author", "cell_type"]]
        .drop_duplicates()
        .set_index("cell_type_author")["cell_type"]
        .to_dict()
    )


ontology_mapping_query = extract_ontology_mapping(adata_query)
ontology_mapping_ref = extract_ontology_mapping(adata_ref)

## Select most similar clusters

In [10]:
top_n_labels = DatasetMapping.select_most_similar_clusters(
    mapping, 
    distance, 
    threshold_mapping=0.25,
    threshold_distance=1.,
    n_top=None
)
top_n_labels

{'ASDC': ['DC'],
 'BFU-E': [],
 'Basophilic Erythroblast': [],
 'CD14 Mono': ['CD14+_Monocyte'],
 'CD16 Mono': ['CD16+_Monocyte'],
 'CD4 Central Memory': ['CD4+_T_cm'],
 'CD4 Effector Memory': ['CD4+_T_em'],
 'CD4 Naive': ['CD4+_T_naive'],
 'CD4 Regulatory': ['Treg'],
 'CD8 Central Memory': ['CD8+_T_GZMK+'],
 'CD8 Effector Memory 1': [],
 'CD8 Effector Memory 2': ['gdT'],
 'CD8 Naive': ['CD8+_T_naive'],
 'CD8 Tissue Resident Memory': ['MAIT'],
 'CFU-E': [],
 'Cycling Progenitor': [],
 'Early GMP': [],
 'Early ProMono': [],
 'EarlyProB': [],
 'EoBasoMast Precursor': [],
 'GMP-Cycle': [],
 'GMP-Mono': [],
 'GMP-Neut': [],
 'HSC': [],
 'Immature B': ['naive_B'],
 'LMPP': [],
 'Large Pre-B': [],
 'Late ProMono': ['Monocyte', 'CD14+_Monocyte'],
 'MEP': [],
 'MLP': [],
 'MLP-II': [],
 'MPP-MkEry': [],
 'MPP-MyLy': [],
 'Mature B': ['IGHMlo_memory_B'],
 'Megakaryocyte': ['Platelet'],
 'Megakaryocyte Precursor': ['Platelet'],
 'NK': ['CD16+_NK'],
 'NK CD56high': ['CD56+_NK'],
 'NK Proliferatin

#### Author provided cluster labels

In [11]:
summaries = []
for i, (k, v) in enumerate(top_n_labels.items()):
    summaries.append(f"*{i+1}*: **{k}**: {v} <br>")

display(Markdown("".join(summaries)))

*1*: **ASDC**: ['DC'] <br>*2*: **BFU-E**: [] <br>*3*: **Basophilic Erythroblast**: [] <br>*4*: **CD14 Mono**: ['CD14+_Monocyte'] <br>*5*: **CD16 Mono**: ['CD16+_Monocyte'] <br>*6*: **CD4 Central Memory**: ['CD4+_T_cm'] <br>*7*: **CD4 Effector Memory**: ['CD4+_T_em'] <br>*8*: **CD4 Naive**: ['CD4+_T_naive'] <br>*9*: **CD4 Regulatory**: ['Treg'] <br>*10*: **CD8 Central Memory**: ['CD8+_T_GZMK+'] <br>*11*: **CD8 Effector Memory 1**: [] <br>*12*: **CD8 Effector Memory 2**: ['gdT'] <br>*13*: **CD8 Naive**: ['CD8+_T_naive'] <br>*14*: **CD8 Tissue Resident Memory**: ['MAIT'] <br>*15*: **CFU-E**: [] <br>*16*: **Cycling Progenitor**: [] <br>*17*: **Early GMP**: [] <br>*18*: **Early ProMono**: [] <br>*19*: **EarlyProB**: [] <br>*20*: **EoBasoMast Precursor**: [] <br>*21*: **GMP-Cycle**: [] <br>*22*: **GMP-Mono**: [] <br>*23*: **GMP-Neut**: [] <br>*24*: **HSC**: [] <br>*25*: **Immature B**: ['naive_B'] <br>*26*: **LMPP**: [] <br>*27*: **Large Pre-B**: [] <br>*28*: **Late ProMono**: ['Monocyte', 'CD14+_Monocyte'] <br>*29*: **MEP**: [] <br>*30*: **MLP**: [] <br>*31*: **MLP-II**: [] <br>*32*: **MPP-MkEry**: [] <br>*33*: **MPP-MyLy**: [] <br>*34*: **Mature B**: ['IGHMlo_memory_B'] <br>*35*: **Megakaryocyte**: ['Platelet'] <br>*36*: **Megakaryocyte Precursor**: ['Platelet'] <br>*37*: **NK**: ['CD16+_NK'] <br>*38*: **NK CD56high**: ['CD56+_NK'] <br>*39*: **NK Proliferating**: ['NK'] <br>*40*: **Orthochromatic Erythroblast**: ['RBC'] <br>*41*: **Plasma Cell**: ['Plasma_B'] <br>*42*: **Polychromatic Erythroblast**: [] <br>*43*: **Pre-ProB**: [] <br>*44*: **Pre-cDC**: [] <br>*45*: **Pre-pDC**: [] <br>*46*: **Pre-pDC Cycling**: [] <br>*47*: **Pro-B Cycling**: [] <br>*48*: **Pro-B VDJ**: [] <br>*49*: **Pro-Erythroblast**: [] <br>*50*: **Small Pre-B**: [] <br>*51*: **Stromal**: [] <br>*52*: **T Proliferating**: [] <br>*53*: **cDC1**: ['cDC1'] <br>*54*: **cDC2**: ['cDC2'] <br>*55*: **pDC**: ['pDC'] <br>

#### Ontology mapped cluster labels

In [12]:
summaries = []
for i, (k, v) in enumerate(top_n_labels.items()):
    summaries.append(
        f"*{i+1}*: **{ontology_mapping_query[k]}**: {[ontology_mapping_ref[elem] for elem in v]} <br>"
    )

display(Markdown("".join(summaries)))

*1*: **dendritic cell**: ['dendritic cell'] <br>*2*: **erythroid progenitor cell**: [] <br>*3*: **basophilic erythroblast**: [] <br>*4*: **CD14-positive monocyte**: ['CD14-positive monocyte'] <br>*5*: **CD14-positive, CD16-positive monocyte**: ['CD14-low, CD16-positive monocyte'] <br>*6*: **central memory CD4-positive, alpha-beta T cell**: ['central memory CD4-positive, alpha-beta T cell'] <br>*7*: **effector memory CD4-positive, alpha-beta T cell**: ['effector memory CD4-positive, alpha-beta T cell'] <br>*8*: **naive thymus-derived CD4-positive, alpha-beta T cell**: ['naive thymus-derived CD4-positive, alpha-beta T cell'] <br>*9*: **CD4-positive, CD25-positive, alpha-beta regulatory T cell**: ['regulatory T cell'] <br>*10*: **central memory CD8-positive, alpha-beta T cell**: ['CD8-positive, alpha-beta memory T cell'] <br>*11*: **effector memory CD8-positive, alpha-beta T cell**: [] <br>*12*: **effector memory CD8-positive, alpha-beta T cell**: ['gamma-delta T cell'] <br>*13*: **naive thymus-derived CD8-positive, alpha-beta T cell**: ['naive thymus-derived CD8-positive, alpha-beta T cell'] <br>*14*: **CD8-positive, alpha-beta memory T cell**: ['mucosal invariant T cell'] <br>*15*: **erythroid progenitor cell**: [] <br>*16*: **common myeloid progenitor**: [] <br>*17*: **granulocyte monocyte progenitor cell**: [] <br>*18*: **promonocyte**: [] <br>*19*: **common lymphoid progenitor**: [] <br>*20*: **basophil mast progenitor cell**: [] <br>*21*: **granulocyte monocyte progenitor cell**: [] <br>*22*: **granulocyte monocyte progenitor cell**: [] <br>*23*: **granulocyte monocyte progenitor cell**: [] <br>*24*: **hematopoietic stem cell**: [] <br>*25*: **immature B cell**: ['naive B cell'] <br>*26*: **hematopoietic oligopotent progenitor cell**: [] <br>*27*: **large pre-B-II cell**: [] <br>*28*: **promonocyte**: ['monocyte', 'CD14-positive monocyte'] <br>*29*: **megakaryocyte-erythroid progenitor cell**: [] <br>*30*: **common lymphoid progenitor**: [] <br>*31*: **common lymphoid progenitor**: [] <br>*32*: **hematopoietic multipotent progenitor cell**: [] <br>*33*: **hematopoietic multipotent progenitor cell**: [] <br>*34*: **mature B cell**: ['memory B cell'] <br>*35*: **megakaryocyte**: ['platelet'] <br>*36*: **megakaryocyte progenitor cell**: ['platelet'] <br>*37*: **natural killer cell**: ['CD16-positive, CD56-dim natural killer cell, human'] <br>*38*: **CD16-negative, CD56-bright natural killer cell, human**: ['CD16-negative, CD56-bright natural killer cell, human'] <br>*39*: **natural killer cell**: ['natural killer cell'] <br>*40*: **orthochromatic erythroblast**: ['erythrocyte'] <br>*41*: **plasma cell**: ['plasma cell'] <br>*42*: **polychromatophilic erythroblast**: [] <br>*43*: **pro-B cell**: [] <br>*44*: **pre-conventional dendritic cell**: [] <br>*45*: **plasmacytoid dendritic cell, human**: [] <br>*46*: **plasmacytoid dendritic cell, human**: [] <br>*47*: **late pro-B cell**: [] <br>*48*: **late pro-B cell**: [] <br>*49*: **erythroid progenitor cell**: [] <br>*50*: **small pre-B-II cell**: [] <br>*51*: **stromal cell of bone marrow**: [] <br>*52*: **T cell**: [] <br>*53*: **conventional dendritic cell**: ['CD141-positive myeloid dendritic cell'] <br>*54*: **conventional dendritic cell**: ['CD1c-positive myeloid dendritic cell'] <br>*55*: **plasmacytoid dendritic cell, human**: ['plasmacytoid dendritic cell'] <br>

## Evaluate cluster similarity

In [13]:
%autoreload
from sconnect.de_testing.selection import (
    select_and_combine_de_results,
    sort_and_filter_de_genes_ova,
    sort_and_filter_de_genes_ava,
)

In [14]:
de_res = {}
for name, adata in [
    ("query", adata_query), 
    ("ref", adata_ref)
]:
    de_res[name] = {}
    for filtering in ["standard", "strict-villani-immune", "strict-villani-bonemarrow"]:
        de_res[name][filtering] = {}
        for selection, n_ova, n_ava in [("top_DE_genes", 10, 3), ("all_DE_genes", None, None)]:
            res = select_and_combine_de_results(
                sort_and_filter_de_genes_ova(
                    adata.uns["de_res_ova"],
                    gene_filtering=filtering,
                ),
                sort_and_filter_de_genes_ava(
                    adata.uns["de_res_ava"],
                    gene_filtering=filtering,
                ),
                n_genes_ova=n_ova,
                n_genes_ava=n_ava,
                overlap_threshold=0.1,
                overlap_n_genes=10,
            )
            for ct in res:  
                res[ct].index.name = "gene"
                res[ct] = res[ct].reset_index().rename(columns={"index": "gene"})
            de_res[name][filtering][selection] = res


In [15]:
def to_hex(m, val):
    rgba = m.to_rgba(val)
    r, g, b, _ = rgba
    return "#{:02x}{:02x}{:02x}".format(int(r*255), int(g*255), int(b*255))


def style_map_val(v, props=""):
    m = cm.ScalarMappable(
        norm=mpl.colors.Normalize(vmin=0.0, vmax=1.0), 
        cmap=plt.get_cmap("Greens")
    )
    return f"background:{to_hex(m, v)};"


def style_dist_val(v, props=""):
    m = cm.ScalarMappable(
        norm=mpl.colors.Normalize(vmin=0.0, vmax=2.0), 
        cmap=plt.get_cmap("RdYlGn_r")
    )
    return f"background:{to_hex(m, v)};"


def highlight_large(val):
    return "font-size: 30pt"


def style_overlap(v, de_res_top, de_res_all, props=""):
    matches_top = de_res_top.gene.tolist()
    matches_all = de_res_all.gene.tolist()
    if v in matches_top:
        return "color:green;"
    elif v in matches_all:
        return "color:orange;"
    else:
        return "color:red;"


In [16]:
FONT_SIZE = 12


for k, v in top_n_labels.items():
    html = [f"<h1> <b>{k}</b>: </h1>"]
    if v:
        # Add summary for suggestions
        html_suggestions = []
        ct_name, map_vals, dist_vals = [], [], []
        for s in v:
            ct_name.append(s)
            map_vals.append(mapping.loc[k][s])
            dist_vals.append(distance.loc[k][s])
        html_suggestions.append(
            pd.DataFrame({"OT mass": map_vals, "Distance": dist_vals}, index=ct_name)
            .style
            .set_table_styles([
                {"selector": "tbody th", "props": [("font-size", f"{FONT_SIZE}pt")]},
                {"selector": "td", "props": [("font-size", f"{FONT_SIZE}pt")]}
            ])
            .format("{:.2f}")
            .map(style_map_val, subset=["OT mass"])
            .map(style_dist_val, subset=["Distance"])
            .set_table_attributes("style='display:inline'")
            .to_html()
        )
        html_suggestions.append("<br /><br />")
        # Add summary for differentially expressed genes
        html_de_overlap = []
        for de_query, de_ref in [
            (
                de_res["query"]["standard"], 
                de_res["ref"]["standard"]
            ),
            (
                de_res["query"]["strict-villani-immune"], 
                de_res["ref"]["strict-villani-immune"]
            )
        ]:
            html_de_overlap += [
                de_query["top_DE_genes"][k]
                .copy()
                .style
                .set_properties(**{"font-size": f"{FONT_SIZE}pt"})
                .format("{:.2f}", subset=["logFC"])
                .format("{:.3f}", subset=["adj.P.Val"])
                .map(
                    style_overlap, 
                    subset=["gene"], 
                    de_res_top=de_ref["top_DE_genes"][v[0]],
                    de_res_all=de_ref["all_DE_genes"][v[0]]
                )
                .set_table_attributes("style='display:inline'")
                .set_table_styles({
                    "reference": [{"selector": "", "props": [("white-space", "normal"), ("width", "200px")]}]
                })
                .set_properties(subset=["reference"], **{"text-align": "left", "font-size": "8pt"})
                .set_caption(f'<span style="font-size: {FONT_SIZE}pt;">QUERY - {k}</span>')
                .to_html()
            ]
            html_de_overlap += [
                de_ref["top_DE_genes"][cluster]
                .copy()
                .style
                .set_properties(**{"font-size": f"{FONT_SIZE}pt"})
                .set_properties(subset=["reference"], **{"text-align": "left", "font-size": "8pt"})
                .format("{:.2f}", subset=["logFC"])
                .format("{:.3f}", subset=["adj.P.Val"])
                .set_table_attributes("style='display:inline'")
                .set_table_styles({
                    "reference": [{"selector": "", "props": [("white-space", "normal"), ("width", "200px")]}]
                })
                .set_caption(f'<span style="font-size: {FONT_SIZE}pt;">REF - {cluster}</span>')
                .to_html()
                for cluster in v
            ]
            html_de_overlap.append("<br /><br />")

        html = html + html_suggestions + html_de_overlap
    else:
        html = html + ["<b><i>No matches found</i></b>"]

    display_html("".join(html) + "<br>", raw=True)


ASDC : 
 
 
 
   
 OT mass 
 Distance 
 
 
 
 
 DC 
 0.62 
 0.63 
 
 
 
 
 
 QUERY - ASDC 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 ALOX5AP 
 4.07 
 0.000 
 ['cDC1'] 
 
 
 1 
 S100A10 
 4.05 
 0.000 
 ['Pre-pDC', 'pDC', 'all', 'Pre-pDC Cycling'] 
 
 
 2 
 LYZ 
 3.95 
 0.000 
 ['pDC', 'Pre-pDC Cycling', 'all'] 
 
 
 3 
 PPP1R14A 
 3.83 
 0.000 
 ['Pre-pDC', 'cDC1', 'all'] 
 
 
 4 
 SCT 
 3.72 
 0.000 
 ['cDC2', 'cDC1', 'all', 'Pre-cDC'] 
 
 
 5 
 LILRA4 
 3.64 
 0.000 
 ['all', 'cDC2', 'Pre-cDC'] 
 
 
 6 
 IRF8 
 3.47 
 0.000 
 ['all'] 
 
 
 7 
 AXL 
 3.41 
 0.000 
 ['all', 'pDC', 'Pre-pDC', 'Pre-pDC Cycling'] 
 
 
 8 
 CST3 
 3.32 
 0.000 
 ['all'] 
 
 
 9 
 TGFBI 
 3.10 
 0.000 
 ['all'] 
 
 
 10 
 PLD4 
 2.85 
 0.000 
 ['all'] 
 
 
 11 
 TSPAN13 
 2.49 
 0.000 
 ['cDC2'] 
 
 
 12 
 PTGDS 
 2.20 
 0.000 
 ['Pre-cDC'] 
 
 
 

 
 REF - DC 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 LYZ 
 5.42 
 0.000 
 ['pDC', 'all'] 
 
 
 1 
 IFI30 
 4.89 
 0.000 
 ['pDC', 'all'] 
 
 
 2 
 CST3 
 4.54 
 0.000 
 ['all'] 
 
 
 3 
 LILRA4 
 4.31 
 0.000 
 ['CD14+_Monocyte', 'Monocyte', 'CD16+_Monocyte', 'all', 'cDC2', 'cDC1', 'cDC'] 
 
 
 4 
 PPP1R14A 
 4.12 
 0.000 
 ['CD16+_Monocyte', 'CD14+_Monocyte', 'Monocyte', 'all'] 
 
 
 5 
 ITM2C 
 4.00 
 0.000 
 ['CD16+_Monocyte', 'Monocyte'] 
 
 
 6 
 PLAC8 
 3.92 
 0.000 
 ['cDC1', 'cDC'] 
 
 
 7 
 ALOX5AP 
 3.91 
 0.000 
 ['cDC1'] 
 
 
 8 
 C12orf75 
 3.85 
 0.000 
 ['CD14+_Monocyte'] 
 
 
 9 
 COTL1 
 3.77 
 0.000 
 ['pDC'] 
 
 
 10 
 PLD4 
 3.65 
 0.000 
 ['all'] 
 
 
 11 
 TGFBI 
 3.43 
 0.000 
 ['all'] 
 
 
 12 
 SIGLEC6 
 3.41 
 0.000 
 ['all', 'cDC'] 
 
 
 13 
 ALDH2 
 3.35 
 0.000 
 ['all'] 
 
 
 14 
 KLF4 
 3.28 
 0.000 
 ['all'] 
 
 
 15 
 PTGDS 
 2.99 
 0.000 
 ['cDC2'] 
 
 
 16 
 LTK 
 2.92 
 0.000 
 ['cDC2'] 
 
 
 
 
 
 QUERY - ASDC 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 LYZ 
 4.52 
 0.000 
 ['MLP-II', 'pDC', 'Pre-pDC Cycling', 'all', 'Pre-pDC'] 
 
 
 1 
 LILRA4 
 3.90 
 0.000 
 ['Megakaryocyte Precursor', 'Early ProMono', 'Late ProMono', 'all', 'MLP-II', 'cDC2', 'CD14 Mono', 'Pre-cDC', 'Pre-pDC', 'cDC1'] 
 
 
 2 
 HLA-DRA 
 3.83 
 0.000 
 ['Megakaryocyte Precursor', 'all'] 
 
 
 3 
 CST3 
 3.73 
 0.000 
 ['MLP-II', 'all'] 
 
 
 4 
 AXL 
 3.54 
 0.000 
 ['Megakaryocyte Precursor', 'Late ProMono', 'Early ProMono', 'CD14 Mono', 'CD16 Mono', 'all', 'pDC', 'Pre-pDC', 'Pre-pDC Cycling', 'cDC1', 'cDC2', 'Pre-cDC'] 
 
 
 5 
 STMN1 
 3.28 
 0.000 
 ['CD16 Mono'] 
 
 
 6 
 SPIB 
 3.17 
 0.000 
 ['Late ProMono', 'Early ProMono', 'CD14 Mono', 'CD16 Mono', 'all', 'Pre-cDC'] 
 
 
 7 
 ANXA1 
 2.74 
 0.000 
 ['pDC'] 
 
 
 8 
 DAB2 
 2.59 
 0.000 
 ['cDC1', 'all'] 
 
 
 9 
 CLEC10A 
 2.56 
 0.000 
 ['Pre-pDC Cycling', 'all'] 
 
 
 10 
 SOX4 
 2.39 
 0.000 
 ['cDC2'] 
 
 
 11 
 FCER1A 
 2.03 
 0.000 
 ['all'] 
 
 
 12 
 IFI30 
 1.93 
 0.000 
 ['all'] 
 
 
 

 
 REF - DC 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 HLA-DRA 
 5.54 
 0.000 
 ['Platelet', 'all'] 
 
 
 1 
 LYZ 
 5.42 
 0.000 
 ['pDC', 'all'] 
 
 
 2 
 HLA-DRB1 
 5.22 
 0.000 
 ['Platelet'] 
 
 
 3 
 IFI30 
 4.89 
 0.000 
 ['pDC', 'Platelet', 'all'] 
 
 
 4 
 CST3 
 4.54 
 0.000 
 ['all'] 
 
 
 5 
 LILRA4 
 4.31 
 0.000 
 ['CD14+_Monocyte', 'Monocyte', 'CD16+_Monocyte', 'all', 'cDC2', 'cDC1', 'cDC'] 
 
 
 6 
 ANXA1 
 3.74 
 0.000 
 ['pDC'] 
 
 
 7 
 SIGLEC6 
 3.71 
 0.000 
 ['Monocyte', 'CD16+_Monocyte', 'CD14+_Monocyte', 'cDC1', 'all', 'cDC', 'cDC2'] 
 
 
 8 
 CLEC10A 
 3.35 
 0.000 
 ['cDC1', 'all'] 
 
 
 9 
 SPIB 
 3.08 
 0.000 
 ['CD14+_Monocyte', 'CD16+_Monocyte', 'Monocyte'] 
 
 
 10 
 DAB2 
 3.00 
 0.000 
 ['all'] 
 
 
 11 
 AXL 
 2.89 
 0.000 
 ['all', 'cDC'] 
 
 
 12 
 CLEC4C 
 2.62 
 0.000 
 ['all', 'cDC2']

BFU-E : No matches found

Basophilic Erythroblast : No matches found

CD14 Mono : 
 
 
 
   
 OT mass 
 Distance 
 
 
 
 
 CD14+_Monocyte 
 0.56 
 0.57 
 
 
 
 
 
 QUERY - CD14 Mono 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 S100A9 
 5.28 
 0.000 
 ['all'] 
 
 
 1 
 S100A8 
 5.26 
 0.000 
 ['all', 'Pre-cDC', 'CD16 Mono', 'cDC2'] 
 
 
 2 
 LYZ 
 4.66 
 0.000 
 ['all'] 
 
 
 3 
 S100A12 
 4.59 
 0.000 
 ['all', 'GMP-Mono', 'CD16 Mono', 'Pre-cDC', 'cDC2'] 
 
 
 4 
 FCN1 
 4.40 
 0.000 
 ['GMP-Mono', 'all'] 
 
 
 5 
 VCAN 
 4.19 
 0.000 
 ['all', 'CD16 Mono'] 
 
 
 6 
 CSTA 
 4.04 
 0.000 
 ['all'] 
 
 
 7 
 MNDA 
 3.69 
 0.000 
 ['all'] 
 
 
 8 
 CD14 
 3.66 
 0.000 
 ['all', 'Pre-cDC'] 
 
 
 9 
 G0S2 
 3.63 
 0.000 
 ['GMP-Mono', 'Early ProMono', 'Late ProMono'] 
 
 
 10 
 TYROBP 
 3.46 
 0.000 
 ['all'] 
 
 
 11 
 RBP7 
 2.57 
 0.000 
 ['cDC2', 'Early ProMono'] 
 
 
 12 
 LGALS2 
 2.43 
 0.000 
 ['Early ProMono', 'Late ProMono'] 
 
 
 13 
 CD300E 
 2.00 
 0.000 
 ['Late ProMono'] 
 
 
 

 
 REF - CD14+_Monocyte 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 S100A8 
 6.80 
 0.000 
 ['cDC1', 'DC', 'all', 'cDC', 'CD16+_Monocyte'] 
 
 
 1 
 S100A9 
 5.85 
 0.000 
 ['DC', 'cDC1', 'all', 'cDC'] 
 
 
 2 
 FCN1 
 5.27 
 0.000 
 ['cDC1', 'DC', 'all'] 
 
 
 3 
 LYZ 
 4.98 
 0.000 
 ['all'] 
 
 
 4 
 VCAN 
 4.49 
 0.000 
 ['all', 'CD16+_Monocyte'] 
 
 
 5 
 S100A12 
 4.43 
 0.000 
 ['all', 'CD16+_Monocyte', 'cDC'] 
 
 
 6 
 CD14 
 4.06 
 0.000 
 ['all'] 
 
 
 7 
 CST3 
 3.90 
 0.000 
 ['all'] 
 
 
 8 
 TNFAIP2 
 3.86 
 0.000 
 ['all'] 
 
 
 9 
 SERPINA1 
 3.83 
 0.000 
 ['all'] 
 
 
 10 
 RBP7 
 2.82 
 0.000 
 ['cDC2'] 
 
 
 11 
 C5AR1 
 2.48 
 0.000 
 ['cDC2'] 
 
 
 12 
 CDA 
 2.48 
 0.000 
 ['cDC2'] 
 
 
 
 
 
 QUERY - CD14 Mono 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 S100A9 
 6.15 
 0.000 
 ['ASDC', 'all', 'cDC1', 'Pre-cDC', 'CD16 Mono', 'cDC2'] 
 
 
 1 
 S100A8 
 6.08 
 0.000 
 ['ASDC', 'cDC1', 'all', 'Pre-cDC', 'CD16 Mono', 'cDC2'] 
 
 
 2 
 FCN1 
 5.01 
 0.000 
 ['cDC1', 'ASDC', 'GMP-Mono', 'all'] 
 
 
 3 
 LYZ 
 4.66 
 0.000 
 ['all'] 
 
 
 4 
 VCAN 
 4.19 
 0.000 
 ['all', 'GMP-Mono', 'CD16 Mono'] 
 
 
 5 
 CD14 
 3.66 
 0.000 
 ['all', 'GMP-Mono', 'Pre-cDC', 'cDC2', 'Early ProMono'] 
 
 
 6 
 SERPINA1 
 3.38 
 0.000 
 ['all', 'Early ProMono', 'Late ProMono'] 
 
 
 7 
 CST3 
 2.81 
 0.000 
 ['all'] 
 
 
 8 
 IFI30 
 2.73 
 0.000 
 ['all', 'Early ProMono'] 
 
 
 9 
 LILRB2 
 2.12 
 0.000 
 ['all'] 
 
 
 10 
 IFITM3 
 1.63 
 0.000 
 ['Late ProMono'] 
 
 
 11 
 CLEC10A 
 1.27 
 0.000 
 ['Late ProMono'] 
 
 
 

 
 REF - CD14+_Monocyte 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 S100A8 
 6.80 
 0.000 
 ['cDC1', 'DC', 'all', 'cDC', 'CD16+_Monocyte', 'cDC2'] 
 
 
 1 
 S100A9 
 5.85 
 0.000 
 ['DC', 'cDC1', 'all', 'cDC', 'CD16+_Monocyte', 'cDC2'] 
 
 
 2 
 FCN1 
 5.27 
 0.000 
 ['cDC1', 'DC', 'all'] 
 
 
 3 
 LYZ 
 4.98 
 0.000 
 ['all'] 
 
 
 4 
 VCAN 
 4.49 
 0.000 
 ['all', 'CD16+_Monocyte'] 
 
 
 5 
 CD14 
 4.06 
 0.000 
 ['all', 'cDC', 'cDC2'] 
 
 
 6 
 CST3 
 3.90 
 0.000 
 ['all'] 
 
 
 7 
 SERPINA1 
 3.83 
 0.000 
 ['all'] 
 
 
 8 
 IFI30 
 3.83 
 0.000 
 ['all'] 
 
 
 9 
 LILRB2 
 2.98 
 0.000 
 ['all']

CD16 Mono : 
 
 
 
   
 OT mass 
 Distance 
 
 
 
 
 CD16+_Monocyte 
 0.62 
 0.46 
 
 
 
 
 
 QUERY - CD16 Mono 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 FCGR3A 
 4.87 
 0.000 
 ['all', 'CD14 Mono'] 
 
 
 1 
 SERPINA1 
 4.62 
 0.000 
 ['all'] 
 
 
 2 
 FCN1 
 4.15 
 0.000 
 ['all'] 
 
 
 3 
 MS4A7 
 3.95 
 0.000 
 ['all'] 
 
 
 4 
 FCER1G 
 3.89 
 0.000 
 ['all'] 
 
 
 5 
 TYROBP 
 3.72 
 0.000 
 ['all'] 
 
 
 6 
 IFI30 
 3.72 
 0.000 
 ['all'] 
 
 
 7 
 LST1 
 3.71 
 0.000 
 ['all'] 
 
 
 8 
 CD68 
 3.66 
 0.000 
 ['all'] 
 
 
 9 
 PILRA 
 3.63 
 0.000 
 ['all'] 
 
 
 10 
 CDKN1C 
 3.11 
 0.000 
 ['CD14 Mono'] 
 
 
 11 
 RHOC 
 3.11 
 0.000 
 ['CD14 Mono'] 
 
 
 

 
 REF - CD16+_Monocyte 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 FCGR3A 
 5.36 
 0.000 
 ['cDC', 'DC', 'cDC2', 'all', 'CD14+_Monocyte', 'Monocyte'] 
 
 
 1 
 SERPINA1 
 5.03 
 0.000 
 ['DC', 'all'] 
 
 
 2 
 CDKN1C 
 4.80 
 0.000 
 ['DC', 'cDC', 'cDC2', 'all', 'CD14+_Monocyte', 'Monocyte'] 
 
 
 3 
 LST1 
 4.40 
 0.000 
 ['all'] 
 
 
 4 
 AIF1 
 4.24 
 0.000 
 ['all'] 
 
 
 5 
 IFI30 
 4.20 
 0.000 
 ['all'] 
 
 
 6 
 MS4A7 
 4.08 
 0.000 
 ['all'] 
 
 
 7 
 FCN1 
 4.05 
 0.000 
 ['all'] 
 
 
 8 
 CST3 
 4.04 
 0.000 
 ['all'] 
 
 
 9 
 C5AR1 
 4.01 
 0.000 
 ['cDC'] 
 
 
 10 
 LILRB2 
 3.97 
 0.000 
 ['all'] 
 
 
 11 
 TCF7L2 
 3.26 
 0.000 
 ['cDC2'] 
 
 
 12 
 CKB 
 2.79 
 0.000 
 ['CD14+_Monocyte'] 
 
 
 13 
 C1QA 
 1.85 
 0.000 
 ['Monocyte'] 
 
 
 
 
 
 QUERY - CD16 Mono 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 FCGR3A 
 5.25 
 0.000 
 ['ASDC', 'cDC1', 'Pre-cDC', 'GMP-Mono', 'Late ProMono', 'Early ProMono', 'cDC2', 'all', 'CD14 Mono'] 
 
 
 1 
 SERPINA1 
 4.84 
 0.000 
 ['ASDC', 'all', 'GMP-Mono', 'Pre-cDC', 'Early ProMono', 'cDC2'] 
 
 
 2 
 FCN1 
 4.83 
 0.000 
 ['cDC1', 'GMP-Mono', 'all'] 
 
 
 3 
 IFITM3 
 4.78 
 0.000 
 ['ASDC', 'cDC1', 'Early ProMono', 'Pre-cDC', 'Late ProMono', 'all', 'cDC2', 'CD14 Mono'] 
 
 
 4 
 IFI30 
 3.72 
 0.000 
 ['all'] 
 
 
 5 
 LILRB2 
 3.39 
 0.000 
 ['all'] 
 
 
 6 
 CST3 
 2.93 
 0.000 
 ['all'] 
 
 
 7 
 SIGLEC10 
 2.73 
 0.000 
 ['Late ProMono', 'all', 'CD14 Mono'] 
 
 
 8 
 S100A9 
 2.70 
 0.000 
 ['all'] 
 
 
 9 
 LYZ 
 2.42 
 0.000 
 ['all'] 
 
 
 

 
 REF - CD16+_Monocyte 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 FCGR3A 
 5.36 
 0.000 
 ['cDC', 'cDC1', 'DC', 'cDC2', 'all', 'CD14+_Monocyte', 'Monocyte'] 
 
 
 1 
 IFITM3 
 5.23 
 0.000 
 ['cDC1', 'DC', 'all', 'cDC'] 
 
 
 2 
 SERPINA1 
 5.03 
 0.000 
 ['DC', 'all', 'cDC'] 
 
 
 3 
 FCN1 
 4.80 
 0.000 
 ['cDC1', 'all'] 
 
 
 4 
 IFI30 
 4.20 
 0.000 
 ['all'] 
 
 
 5 
 CST3 
 4.04 
 0.000 
 ['all'] 
 
 
 6 
 LILRB2 
 3.97 
 0.000 
 ['all'] 
 
 
 7 
 SIGLEC10 
 2.90 
 0.000 
 ['all', 'cDC2', 'CD14+_Monocyte', 'Monocyte'] 
 
 
 8 
 LYZ 
 2.80 
 0.000 
 ['all'] 
 
 
 9 
 S100A9 
 2.67 
 0.000 
 ['all'] 
 
 
 10 
 CD79B 
 2.12 
 0.000 
 ['CD14+_Monocyte', 'cDC2', 'Monocyte']

...

Stromal : No matches found

T Proliferating : No matches found

cDC1 : 
 
 
 
   
 OT mass 
 Distance 
 
 
 
 
 cDC1 
 0.62 
 0.44 
 
 
 
 
 
 QUERY - cDC1 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 CST3 
 3.47 
 0.000 
 ['all'] 
 
 
 1 
 LYZ 
 3.39 
 0.000 
 ['all'] 
 
 
 2 
 HLA-DQA1 
 3.39 
 0.000 
 ['all'] 
 
 
 3 
 IRF8 
 3.28 
 0.000 
 ['all'] 
 
 
 4 
 HLA-DQB1 
 3.12 
 0.000 
 ['all'] 
 
 
 5 
 CLEC9A 
 3.06 
 0.000 
 ['ASDC', 'cDC2', 'Pre-cDC'] 
 
 
 6 
 HLA-DPA1 
 2.94 
 0.000 
 ['all'] 
 
 
 7 
 C1orf54 
 2.92 
 0.000 
 ['all', 'ASDC', 'Pre-cDC'] 
 
 
 8 
 CPVL 
 2.92 
 0.000 
 ['all'] 
 
 
 9 
 LGALS2 
 2.90 
 0.000 
 ['all', 'ASDC'] 
 
 
 10 
 HLA-DPB1 
 2.80 
 0.000 
 ['all'] 
 
 
 11 
 DNASE1L3 
 2.26 
 0.000 
 ['Pre-cDC', 'cDC2'] 
 
 
 12 
 CLNK 
 2.17 
 0.000 
 ['cDC2'] 
 
 
 

 
 REF - cDC1 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 CST3 
 7.01 
 0.000 
 ['atypical_B', 'IGHMlo_memory_B', 'all'] 
 
 
 1 
 CPVL 
 5.65 
 0.000 
 ['atypical_B', 'IGHMlo_memory_B', 'all', 'DC'] 
 
 
 2 
 LYZ 
 5.49 
 0.000 
 ['atypical_B', 'IGHMlo_memory_B', 'all'] 
 
 
 3 
 CLEC9A 
 4.17 
 0.000 
 ['DC', 'all', 'Monocyte', 'CD14+_Monocyte', 'cDC2', 'cDC'] 
 
 
 4 
 C1orf54 
 4.01 
 0.000 
 ['DC', 'all', 'Monocyte', 'CD14+_Monocyte', 'cDC2', 'cDC'] 
 
 
 5 
 HLA-DRA 
 3.95 
 0.000 
 ['all'] 
 
 
 6 
 DNASE1L3 
 3.80 
 0.000 
 ['Monocyte', 'CD14+_Monocyte', 'all', 'cDC2', 'cDC'] 
 
 
 7 
 LGALS2 
 3.73 
 0.000 
 ['all'] 
 
 
 8 
 HLA-DQA1 
 3.71 
 0.000 
 ['all'] 
 
 
 9 
 IRF8 
 3.67 
 0.000 
 ['all'] 
 
 
 
 
 
 QUERY - cDC1 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 LYZ 
 5.08 
 0.000 
 ['MLP-II', 'all'] 
 
 
 1 
 CST3 
 3.89 
 0.000 
 ['MLP-II', 'all'] 
 
 
 2 
 CLEC9A 
 3.06 
 0.000 
 ['ASDC', 'CD16 Mono', 'CD14 Mono', 'Late ProMono', 'Early ProMono', 'MLP-II', 'cDC2', 'all', 'Pre-cDC'] 
 
 
 3 
 HLA-DRA 
 2.70 
 0.000 
 ['all', 'Late ProMono', 'Early ProMono'] 
 
 
 4 
 HLA-DRB1 
 2.69 
 0.000 
 ['all'] 
 
 
 5 
 STMN1 
 2.53 
 0.000 
 ['CD16 Mono', 'CD14 Mono'] 
 
 
 6 
 CLNK 
 2.27 
 0.000 
 ['CD14 Mono', 'CD16 Mono', 'Late ProMono', 'Early ProMono', 'cDC2', 'all', 'ASDC', 'Pre-cDC'] 
 
 
 7 
 HAVCR2 
 1.67 
 0.000 
 ['all'] 
 
 
 8 
 CD83 
 1.67 
 0.000 
 ['ASDC', 'all'] 
 
 
 9 
 SPINK2 
 1.36 
 0.000 
 ['cDC2'] 
 
 
 10 
 CXCR3 
 1.27 
 0.000 
 ['all', 'Pre-cDC'] 
 
 
 11 
 IFI30 
 1.10 
 0.000 
 ['all'] 
 
 
 

 
 REF - cDC1 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 CST3 
 7.01 
 0.000 
 ['atypical_B', 'all'] 
 
 
 1 
 HLA-DRA 
 6.39 
 0.000 
 ['Platelet', 'all'] 
 
 
 2 
 HLA-DRB1 
 6.22 
 0.000 
 ['Platelet', 'all'] 
 
 
 3 
 LYZ 
 5.49 
 0.000 
 ['atypical_B', 'Platelet', 'all'] 
 
 
 4 
 CLEC9A 
 4.17 
 0.000 
 ['DC', 'atypical_B', 'CD16+_Monocyte', 'all', 'Monocyte', 'CD14+_Monocyte', 'cDC2', 'cDC'] 
 
 
 5 
 CLNK 
 3.14 
 0.000 
 ['CD16+_Monocyte', 'CD14+_Monocyte', 'Monocyte', 'DC', 'all', 'cDC2', 'cDC'] 
 
 
 6 
 ACTN1 
 2.93 
 0.000 
 ['DC', 'all'] 
 
 
 7 
 ZNF366 
 2.37 
 0.000 
 ['CD16+_Monocyte', 'all', 'Monocyte', 'CD14+_Monocyte', 'cDC'] 
 
 
 8 
 HAVCR2 
 1.94 
 0.000 
 ['all'] 
 
 
 9 
 CXCR3 
 1.85 
 0.000 
 ['cDC2'] 
 
 
 10 
 CD83 
 1.82 
 0.000 
 ['all']

cDC2 : 
 
 
 
   
 OT mass 
 Distance 
 
 
 
 
 cDC2 
 0.62 
 0.45 
 
 
 
 
 
 QUERY - cDC2 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 LYZ 
 4.05 
 0.000 
 ['all'] 
 
 
 1 
 S100A9 
 3.82 
 0.000 
 ['ASDC'] 
 
 
 2 
 FCER1A 
 3.72 
 0.000 
 ['Early ProMono', 'all', 'GMP-Mono', 'CD14 Mono', 'Late ProMono', 'cDC1'] 
 
 
 3 
 CLEC10A 
 3.65 
 0.000 
 ['cDC1', 'Early ProMono', 'all', 'Late ProMono', 'GMP-Mono', 'Pre-cDC'] 
 
 
 4 
 FCN1 
 3.39 
 0.000 
 ['cDC1', 'ASDC'] 
 
 
 5 
 CSTA 
 3.38 
 0.000 
 ['all'] 
 
 
 6 
 HLA-DQA1 
 3.34 
 0.000 
 ['Late ProMono', 'Early ProMono', 'all'] 
 
 
 7 
 CST3 
 3.30 
 0.000 
 ['all'] 
 
 
 8 
 S100A8 
 3.23 
 0.000 
 ['ASDC'] 
 
 
 9 
 LGALS2 
 3.19 
 0.000 
 ['all', 'Pre-cDC'] 
 
 
 10 
 MNDA 
 3.08 
 0.000 
 ['all'] 
 
 
 11 
 CD1C 
 3.03 
 0.000 
 ['GMP-Mono', 'all', 'CD14 Mono', 'Pre-cDC'] 
 
 
 12 
 CFP 
 3.01 
 0.000 
 ['all'] 
 
 
 13 
 ENHO 
 2.82 
 0.000 
 ['CD14 Mono'] 
 
 
 

 
 REF - cDC2 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 CST3 
 4.71 
 0.000 
 ['all'] 
 
 
 1 
 LYZ 
 4.69 
 0.000 
 ['all'] 
 
 
 2 
 S100A8 
 4.54 
 0.000 
 ['cDC1', 'cDC'] 
 
 
 3 
 FCER1A 
 4.31 
 0.000 
 ['CD16+_Monocyte', 'cDC1', 'CD14+_Monocyte', 'all', 'Monocyte'] 
 
 
 4 
 FCN1 
 4.25 
 0.000 
 ['cDC1', 'DC'] 
 
 
 5 
 S100A9 
 4.12 
 0.000 
 ['DC', 'all', 'cDC'] 
 
 
 6 
 CLEC10A 
 4.00 
 0.000 
 ['all', 'CD16+_Monocyte', 'Monocyte'] 
 
 
 7 
 CPVL 
 3.76 
 0.000 
 ['all'] 
 
 
 8 
 CD1C 
 3.67 
 0.000 
 ['CD16+_Monocyte', 'DC', 'Monocyte', 'CD14+_Monocyte'] 
 
 
 9 
 IL1B 
 3.65 
 0.000 
 ['all'] 
 
 
 10 
 IFI30 
 3.64 
 0.000 
 ['all'] 
 
 
 11 
 HLA-DRA 
 3.64 
 0.000 
 ['all'] 
 
 
 12 
 LGALS2 
 3.62 
 0.000 
 ['all'] 
 
 
 13 
 ENHO 
 3.12 
 0.000 
 ['CD14+_Monocyte'] 
 
 
 14 
 VCAN 
 2.39 
 0.000 
 ['cDC'] 
 
 
 
 
 
 QUERY - cDC2 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 LYZ 
 5.14 
 0.000 
 ['pDC', 'all'] 
 
 
 1 
 FCER1A 
 4.46 
 0.000 
 ['CD16 Mono', 'Early ProMono', 'all', 'GMP-Mono', 'CD14 Mono', 'Late ProMono', 'cDC1'] 
 
 
 2 
 S100A9 
 3.82 
 0.000 
 ['ASDC', 'pDC', 'all'] 
 
 
 3 
 CLEC10A 
 3.81 
 0.000 
 ['pDC', 'cDC1', 'Early ProMono', 'all', 'Late ProMono', 'GMP-Mono', 'CD16 Mono', 'CD14 Mono', 'Pre-cDC'] 
 
 
 4 
 FCN1 
 3.39 
 0.000 
 ['cDC1', 'ASDC', 'all'] 
 
 
 5 
 CST3 
 3.30 
 0.000 
 ['all'] 
 
 
 6 
 S100A8 
 3.23 
 0.000 
 ['ASDC'] 
 
 
 7 
 CD1C 
 3.03 
 0.000 
 ['GMP-Mono', 'Late ProMono', 'Early ProMono', 'all', 'CD16 Mono', 'CD14 Mono', 'Pre-cDC'] 
 
 
 8 
 IFI30 
 2.84 
 0.000 
 ['all'] 
 
 
 9 
 HLA-DRA 
 2.55 
 0.000 
 ['all'] 
 
 
 10 
 HLA-DRB1 
 2.49 
 0.000 
 ['all'] 
 
 
 11 
 IFITM3 
 1.24 
 0.000 
 ['Pre-cDC'] 
 
 
 

 
 REF - cDC2 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 CST3 
 4.71 
 0.000 
 ['all'] 
 
 
 1 
 LYZ 
 4.69 
 0.000 
 ['all'] 
 
 
 2 
 S100A8 
 4.54 
 0.000 
 ['cDC1', 'cDC'] 
 
 
 3 
 FCER1A 
 4.31 
 0.000 
 ['CD16+_Monocyte', 'cDC1', 'CD14+_Monocyte', 'all', 'Monocyte'] 
 
 
 4 
 FCN1 
 4.25 
 0.000 
 ['cDC1', 'DC', 'all'] 
 
 
 5 
 S100A9 
 4.12 
 0.000 
 ['DC', 'all', 'cDC'] 
 
 
 6 
 CLEC10A 
 4.00 
 0.000 
 ['all', 'CD16+_Monocyte', 'Monocyte', 'CD14+_Monocyte'] 
 
 
 7 
 CD1C 
 3.67 
 0.000 
 ['CD16+_Monocyte', 'DC', 'all', 'Monocyte', 'CD14+_Monocyte'] 
 
 
 8 
 IFI30 
 3.64 
 0.000 
 ['all'] 
 
 
 9 
 HLA-DRA 
 3.64 
 0.000 
 ['all'] 
 
 
 10 
 VCAN 
 3.43 
 0.000 
 ['all', 'cDC']

pDC : 
 
 
 
   
 OT mass 
 Distance 
 
 
 
 
 pDC 
 0.62 
 0.78 
 
 
 
 
 
 QUERY - pDC 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 GZMB 
 4.85 
 0.000 
 ['Pre-pDC', 'all', 'ASDC', 'Pre-pDC Cycling'] 
 
 
 1 
 LILRA4 
 4.14 
 0.000 
 ['all', 'Pre-pDC'] 
 
 
 2 
 SCT 
 4.00 
 0.000 
 ['all'] 
 
 
 3 
 IRF7 
 3.79 
 0.000 
 ['all'] 
 
 
 4 
 SERPINF1 
 3.69 
 0.000 
 ['all', 'Pre-pDC Cycling'] 
 
 
 5 
 IRF8 
 3.67 
 0.000 
 ['all'] 
 
 
 6 
 PLD4 
 3.59 
 0.000 
 ['all'] 
 
 
 7 
 ALOX5AP 
 3.53 
 0.000 
 ['Pre-pDC'] 
 
 
 8 
 UGCG 
 3.35 
 0.000 
 ['all'] 
 
 
 9 
 SMPD3 
 3.31 
 0.000 
 ['all', 'ASDC'] 
 
 
 10 
 TPM2 
 3.31 
 0.000 
 ['all'] 
 
 
 11 
 PTGDS 
 3.20 
 0.000 
 ['Pre-pDC Cycling'] 
 
 
 12 
 RASD1 
 2.81 
 0.000 
 ['ASDC'] 
 
 
 

 
 REF - pDC 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 PLD4 
 5.55 
 0.000 
 ['CD16+_NK', 'all'] 
 
 
 1 
 ITM2C 
 5.23 
 0.000 
 ['CD16+_NK'] 
 
 
 2 
 LILRA4 
 5.05 
 0.000 
 ['CD16+_NK', 'all'] 
 
 
 3 
 GZMB 
 4.80 
 0.000 
 ['DC', 'all'] 
 
 
 4 
 PTGDS 
 4.64 
 0.000 
 ['all'] 
 
 
 5 
 SERPINF1 
 4.53 
 0.000 
 ['all'] 
 
 
 6 
 IRF8 
 4.19 
 0.000 
 ['all'] 
 
 
 7 
 TCF4 
 4.12 
 0.000 
 ['all'] 
 
 
 8 
 CLEC4C 
 4.11 
 0.000 
 ['all'] 
 
 
 9 
 SCT 
 4.08 
 0.000 
 ['all'] 
 
 
 10 
 LRRC26 
 3.96 
 0.000 
 ['all'] 
 
 
 11 
 NIBAN3 
 3.59 
 0.000 
 ['DC'] 
 
 
 12 
 DERL3 
 3.48 
 0.000 
 ['DC'] 
 
 
 
 
 
 QUERY - pDC 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 LILRA4 
 4.42 
 0.000 
 ['EarlyProB', 'Pre-ProB', 'Small Pre-B', 'Plasma Cell', 'all', 'Immature B', 'MLP-II', 'cDC2', 'Pre-cDC', 'Pre-pDC'] 
 
 
 1 
 CST3 
 4.07 
 0.000 
 ['Small Pre-B', 'Immature B', 'Pre-ProB', 'all'] 
 
 
 2 
 MZB1 
 4.06 
 0.000 
 ['cDC2', 'Pre-cDC', 'all', 'ASDC'] 
 
 
 3 
 FCER1A 
 3.78 
 0.000 
 ['Immature B', 'Pre-ProB', 'Small Pre-B', 'EarlyProB', 'Plasma Cell', 'MLP-II', 'all'] 
 
 
 4 
 SOX4 
 3.77 
 0.000 
 ['Plasma Cell'] 
 
 
 5 
 DERL3 
 3.44 
 0.000 
 ['cDC2', 'all', 'ASDC'] 
 
 
 6 
 SPIB 
 3.38 
 0.000 
 ['EarlyProB', 'all'] 
 
 
 7 
 CLEC4C 
 3.12 
 0.000 
 ['MLP-II', 'Pre-pDC', 'all'] 
 
 
 8 
 NIBAN3 
 3.06 
 0.000 
 ['Pre-cDC', 'all'] 
 
 
 9 
 TCL1A 
 2.81 
 0.000 
 ['Pre-pDC', 'ASDC', 'all', 'Pre-pDC Cycling'] 
 
 
 10 
 IRF4 
 2.61 
 0.000 
 ['all', 'Pre-pDC Cycling'] 
 
 
 11 
 CD4 
 1.85 
 0.000 
 ['Pre-pDC Cycling'] 
 
 
 

 
 REF - pDC 
 
 
   
 gene 
 logFC 
 adj.P.Val 
 reference 
 
 
 
 
 0 
 CST3 
 5.12 
 0.000 
 ['Plasma_B', 'IGHMhi_memory_B', 'IGHMlo_memory_B', 'naive_B', 'all'] 
 
 
 1 
 HLA-DRA 
 5.06 
 0.000 
 ['Platelet'] 
 
 
 2 
 LILRA4 
 5.05 
 0.000 
 ['Platelet', 'Plasma_B', 'all', 'IGHMlo_memory_B', 'naive_B', 'IGHMhi_memory_B'] 
 
 
 3 
 HLA-DRB1 
 4.69 
 0.000 
 ['Platelet'] 
 
 
 4 
 NIBAN3 
 4.28 
 0.000 
 ['Plasma_B', 'all', 'DC'] 
 
 
 5 
 CLEC4C 
 4.14 
 0.000 
 ['IGHMlo_memory_B', 'all', 'IGHMhi_memory_B', 'naive_B'] 
 
 
 6 
 DERL3 
 3.73 
 0.000 
 ['all', 'DC'] 
 
 
 7 
 MZB1 
 3.72 
 0.000 
 ['all'] 
 
 
 8 
 SPIB 
 3.48 
 0.000 
 ['all'] 
 
 
 9 
 TCL1A 
 3.47 
 0.000 
 ['DC', 'all'] 
 
 
 10 
 IRF4 
 2.80 
 0.000 
 ['all'] 
 
 
 11 
 DAB2 
 2.74 
 0.000 
 ['all']